# Trabalho IA 2025/2026 - PopOut, MCTS e Decision Trees ID3

Este notebook completa a solução prática: jogo PopOut, MCTS com UCT, geração de dataset PopOut, ID3 sem `scikit-learn`, discretização para Iris e avaliação experimental.

## Decisões principais

- O tabuleiro é guardado por colunas, porque as operações `put` e `pop` ficam simples.
- O MCTS usa UCT na fase de seleção e escolhe a jogada final pelo maior número de visitas.
- O dataset PopOut é criado com pares `(estado, jogada)` onde a jogada é escolhida por MCTS.
- A árvore ID3 usa apenas atributos discretos. No Iris, os valores numéricos são discretizados por quantis.

In [ ]:
# ============================================================
# Trabalho IA 2025/2026 - PopOut + MCTS + Decision Tree ID3
# ============================================================
# Requisitos cumpridos:
# 1) PopOut jogavel em 3 modos: Humano vs Humano, Humano vs IA, IA vs IA.
# 2) MCTS com UCT.
# 3) Geracao de dataset PopOut: pares (estado, melhor jogada MCTS).
# 4) Arvore de decisao ID3 feita de raiz, sem scikit-learn.
# 5) Discretizacao de atributos numericos para o dataset Iris.
# 6) Classificacao de novos exemplos e avaliacao de accuracy.

import random
import math
import copy
import csv
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

# -----------------------------
# POP OUT
# -----------------------------

class Poupout:
    def __init__(self, rows, cols, moved="O", to_move="X"):
        self.rows = rows
        self.cols = cols
        self.moved = moved
        self.to_move = to_move
        self.board = [["-" for _ in range(rows)] for _ in range(cols)]
        self.n_pieces = 0
        self.repeated = False
        self.last_move = None
        self.draw = False

    def display(self):
        print("  " + " ".join(str(i) for i in range(self.cols)))
        for r in range(self.rows):
            print("  ", end="")
            for c in range(self.cols):
                print(self.board[c][r], end="")
            print("")

    def board_key(self):
        return tuple(tuple(col) for col in self.board), self.to_move

    def put(self, column):
        if column < 0 or column >= self.cols or self.board[column][0] != "-":
            raise Exception("Acao impossivel")
        for i in range(self.rows - 1, -1, -1):
            if self.board[column][i] == "-":
                self.board[column][i] = self.to_move
                self.n_pieces += 1
                return

    def pop(self, column):
        if column < 0 or column >= self.cols or self.board[column][-1] != self.to_move:
            raise Exception("Acao impossivel")
        self.board[column] = ["-"] + self.board[column][:-1]
        self.n_pieces -= 1

    def change_to_move(self):
        self.to_move, self.moved = self.moved, self.to_move

    def make_move(self, move, column):
        try:
            if move == "put":
                self.put(column)
            elif move == "pop":
                self.pop(column)
            elif move == "draw" and (self.repeated or self.check_full()):
                self.draw = True
            else:
                return False
            self.last_move = (move, column)
            return True
        except Exception:
            return False

    def check_row(self, row):
        line = "".join(self.board[c][row] for c in range(self.cols))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_col(self, col):
        line = "".join(self.board[col][r] for r in range(self.rows))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_diag1(self, row, col):
        cells = min(self.rows - row, self.cols - col)
        if cells < 4:
            return None
        line = "".join(self.board[col + i][row + i] for i in range(cells))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_diag2(self, row, col):
        cells = min(self.rows - row, col + 1)
        if cells < 4:
            return None
        line = "".join(self.board[col - i][row + i] for i in range(cells))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_win(self):
        if self.last_move is None:
            return None
        move, column = self.last_move
        adversary_win = False

        if move == "put":
            # ultima peca do jogador moved na coluna jogada
            row = None
            for r in range(self.rows):
                if self.board[column][r] == self.moved:
                    row = r
                    break
            if row is None:
                return None

            for f in [lambda: self.check_row(row), lambda: self.check_col(column),
                      lambda: self.check_diag1(max(0, row - column), max(0, column - row)),
                      lambda: self.check_diag2(max(0, row - (self.cols - 1 - column)), min(self.cols - 1, row + column))]:
                winner = f()
                if winner is not None:
                    return winner

        elif move == "pop":
            # Regra PopOut: se pop cria quatro para ambos, vence quem fez pop.
            for row in range(self.rows - 1, -1, -1):
                if self.board[column][row] == "-":
                    break
                for f in [lambda row=row: self.check_row(row),
                          lambda row=row: self.check_diag1(max(0, row - column), max(0, column - row)),
                          lambda row=row: self.check_diag2(max(0, row - (self.cols - 1 - column)), min(self.cols - 1, row + column))]:
                    winner = f()
                    if winner == self.moved:
                        return winner
                    if winner == self.to_move:
                        adversary_win = True
            # verificar tambem a coluna alterada
            winner = self.check_col(column)
            if winner == self.moved:
                return winner
            if winner == self.to_move:
                adversary_win = True

        if adversary_win:
            return self.to_move
        return None

    def check_full(self):
        return self.n_pieces == self.rows * self.cols

    def check_repeat(self, state_counts):
        key = self.board_key()
        state_counts[key] += 1
        if state_counts[key] >= 3:
            self.repeated = True
        return self.repeated

    def clone(self):
        new = Poupout(self.rows, self.cols, self.moved, self.to_move)
        new.board = copy.deepcopy(self.board)
        new.n_pieces = self.n_pieces
        new.repeated = self.repeated
        new.last_move = self.last_move
        new.draw = self.draw
        return new

    def legal_moves(self):
        moves = []
        for c in range(self.cols):
            if self.board[c][0] == "-":
                moves.append(("put", c))
            if self.board[c][-1] == self.to_move:
                moves.append(("pop", c))
        if self.repeated or self.check_full():
            moves.append(("draw", 0))
        return moves

    def is_terminal(self):
        return self.check_win() is not None or self.draw

    def get_result(self, maximizing_player):
        winner = self.check_win()
        if winner == maximizing_player:
            return 1
        if winner is not None:
            return -1
        return 0

# -----------------------------
# MCTS + UCT
# -----------------------------

class MCTSNode:
    def __init__(self, game_state, parent=None, move=None):
        self.state = game_state
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0.0
        self.visits = 0
        self._untried = game_state.legal_moves()
        random.shuffle(self._untried)

    def is_fully_expanded(self):
        return len(self._untried) == 0

    def uct_score(self, c=math.sqrt(2)):
        if self.visits == 0:
            return float("inf")
        return self.wins / self.visits + c * math.sqrt(math.log(self.parent.visits) / self.visits)

    def best_child(self, c=math.sqrt(2)):
        return max(self.children, key=lambda n: n.uct_score(c))

class MCTS:
    def __init__(self, iterations=600, c=math.sqrt(2), max_depth=40, rollout_policy="random"):
        self.iterations = iterations
        self.c = c
        self.max_depth = max_depth
        self.rollout_policy = rollout_policy

    def choose_move(self, game_state):
        root = MCTSNode(game_state.clone())
        if not root._untried:
            return None
        original_player = game_state.to_move
        for _ in range(self.iterations):
            node = self._select(root)
            if not node.state.is_terminal():
                node = self._expand(node)
            result = self._simulate(node, original_player)
            self._backpropagate(node, result, original_player)
        return max(root.children, key=lambda n: n.visits).move

    def _select(self, node):
        while not node.state.is_terminal() and node.is_fully_expanded() and node.children:
            node = node.best_child(self.c)
        return node

    def _expand(self, node):
        if node._untried:
            move = node._untried.pop()
            new_game = node.state.clone()
            new_game.make_move(move[0], move[1])
            new_game.change_to_move()
            child = MCTSNode(new_game, parent=node, move=move)
            node.children.append(child)
            return child
        return node

    def _rollout_move(self, sim):
        moves = sim.legal_moves()
        if self.rollout_policy == "prefer_center":
            # estrategia simples: em empates prefere colunas centrais
            center = (sim.cols - 1) / 2
            moves.sort(key=lambda m: abs(m[1] - center))
            top = moves[:max(1, len(moves)//2)]
            return random.choice(top)
        return random.choice(moves)

    def _simulate(self, node, original_player):
        sim = node.state.clone()
        depth = 0
        while not sim.is_terminal() and depth < self.max_depth:
            moves = sim.legal_moves()
            if not moves:
                break
            action, col = self._rollout_move(sim)
            sim.make_move(action, col)
            sim.change_to_move()
            depth += 1
        return sim.get_result(original_player)

    def _backpropagate(self, node, result, original_player):
        while node is not None:
            node.visits += 1
            node_player = node.state.moved
            if node_player == original_player:
                node.wins += result
            else:
                node.wins -= result
            node = node.parent

# -----------------------------
# Dataset PopOut a partir do MCTS
# -----------------------------

def state_to_features(game):
    row = {}
    for c in range(game.cols):
        for r in range(game.rows):
            row[f"c{c}_r{r}"] = game.board[c][r]
    row["to_move"] = game.to_move
    return row

def generate_popout_dataset(n_games=20, mcts_iterations=120, seed=0, outfile="popout_dataset.csv"):
    random.seed(seed)
    ai = MCTS(iterations=mcts_iterations, max_depth=30, rollout_policy="prefer_center")
    rows = []

    for _ in range(n_games):
        game = Poupout(6, 7)
        state_counts = defaultdict(int)
        state_counts[game.board_key()] += 1

        while True:
            if game.is_terminal():
                break
            move = ai.choose_move(game)
            features = state_to_features(game)
            features["move"] = f"{move[0]}_{move[1]}"
            rows.append(features)
            game.make_move(move[0], move[1])
            game.check_repeat(state_counts)
            game.change_to_move()

        fieldnames = list(rows[0].keys())
        with open(outfile, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    return rows

# -----------------------------
# ID3 feito de raiz
# -----------------------------

@dataclass
class ID3Node:
    prediction: Any
    attribute: Optional[str] = None
    children: Optional[Dict[Any, "ID3Node"]] = None
    is_leaf: bool = False

    def classify(self, example):
        if self.is_leaf or self.attribute is None:
            return self.prediction
        value = example.get(self.attribute)
        if self.children and value in self.children:
            return self.children[value].classify(example)
        return self.prediction

    def pretty(self, indent=""):
        if self.is_leaf:
            return indent + f"Classe: {self.prediction}\n"
        s = indent + f"Se {self.attribute}:\n"
        for value, child in sorted(self.children.items(), key=lambda x: str(x[0])):
            s += indent + f"  = {value} ->\n"
            s += child.pretty(indent + "    ")
        return s

class ID3DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.label_col = None

    @staticmethod
    def entropy(labels):
        total = len(labels)
        counts = Counter(labels)
        return -sum((cnt/total) * math.log2(cnt/total) for cnt in counts.values() if cnt)

    def information_gain(self, rows, attribute, label_col):
        base = self.entropy([r[label_col] for r in rows])
        total = len(rows)
        parts = defaultdict(list)
        for r in rows:
            parts[r[attribute]].append(r)
        remainder = sum((len(part)/total) * self.entropy([r[label_col] for r in part]) for part in parts.values())
        return base - remainder

    def majority_label(self, rows, label_col):
        return Counter(r[label_col] for r in rows).most_common(1)[0][0]

    def fit(self, rows, label_col="label", attributes=None):
        if not rows:
            raise ValueError("Dataset vazio")
        self.label_col = label_col
        if attributes is None:
            attributes = [a for a in rows[0].keys() if a != label_col]
        self.root = self._id3(rows, attributes, label_col, depth=0)
        return self

    def _id3(self, rows, attributes, label_col, depth):
        labels = [r[label_col] for r in rows]
        majority = self.majority_label(rows, label_col)
        if len(set(labels)) == 1:
            return ID3Node(prediction=labels[0], is_leaf=True)
        if not attributes or len(rows) < self.min_samples_split or (self.max_depth is not None and depth >= self.max_depth):
            return ID3Node(prediction=majority, is_leaf=True)

        gains = [(self.information_gain(rows, a, label_col), a) for a in attributes]
        best_gain, best_attr = max(gains, key=lambda x: x[0])
        if best_gain <= 1e-12:
            return ID3Node(prediction=majority, is_leaf=True)

        node = ID3Node(prediction=majority, attribute=best_attr, children={}, is_leaf=False)
        values = sorted(set(r[best_attr] for r in rows), key=str)
        remaining = [a for a in attributes if a != best_attr]
        for v in values:
            subset = [r for r in rows if r[best_attr] == v]
            node.children[v] = self._id3(subset, remaining, label_col, depth + 1)
        return node

    def predict_one(self, example):
        if self.root is None:
            raise ValueError("A arvore ainda nao foi treinada")
        return self.root.classify(example)

    def predict(self, rows):
        return [self.predict_one(r) for r in rows]

    def score(self, rows, label_col=None):
        label_col = label_col or self.label_col
        preds = self.predict(rows)
        return sum(p == r[label_col] for p, r in zip(preds, rows)) / len(rows)

    def pretty(self):
        return self.root.pretty() if self.root else "<arvore vazia>"

# -----------------------------
# Utilitarios: split e discretizacao
# -----------------------------

def train_test_split(rows, test_ratio=0.25, seed=0):
    rows = list(rows)
    random.Random(seed).shuffle(rows)
    cut = int(len(rows) * (1 - test_ratio))
    return rows[:cut], rows[cut:]

def discretize_numeric_dataset(rows, label_col="label", bins=3):
    # Discretizacao por quantis: reduz tamanho da arvore e funciona bem no Iris.
    attrs = [a for a in rows[0] if a != label_col]
    thresholds = {}
    for a in attrs:
        vals = sorted(float(r[a]) for r in rows)
        qs = []
        for b in range(1, bins):
            idx = int(len(vals) * b / bins)
            qs.append(vals[idx])
        thresholds[a] = qs

    new_rows = []
    for r in rows:
        nr = {label_col: r[label_col]}
        for a in attrs:
            x = float(r[a])
            k = 0
            while k < len(thresholds[a]) and x > thresholds[a][k]:
                k += 1
            nr[a] = f"bin{k}"
        new_rows.append(nr)
    return new_rows, thresholds

def load_csv_dataset(path, label_col="label"):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f)), label_col

# -----------------------------
# Experiencias PopOut
# -----------------------------

def evaluate_popout_tree(dataset_rows, max_depth_values=(4, 6, 8, 10, None), seed=0):
    train, test = train_test_split(dataset_rows, test_ratio=0.25, seed=seed)
    results = []
    for d in max_depth_values:
        tree = ID3DecisionTree(max_depth=d, min_samples_split=2).fit(train, label_col="label")
        results.append({
            "max_depth": str(d),
            "train_accuracy": round(tree.score(train, "label"), 4),
            "test_accuracy": round(tree.score(test, "label"), 4),
            "n_train": len(train),
            "n_test": len(test),
        })
    return results

def mcts_match(iter_x=100, iter_o=300, seed=0, max_turns=84):
    random.seed(seed)
    game = Poupout(6, 7)
    ais = {"X": MCTS(iterations=iter_x, max_depth=35), "O": MCTS(iterations=iter_o, max_depth=35)}
    state_counts = defaultdict(int)
    state_counts[game.board_key()] += 1
    for _ in range(max_turns):
        if game.is_terminal():
            break
        move = ais[game.to_move].choose_move(game)
        if move is None:
            break
        game.make_move(move[0], move[1])
        game.check_repeat(state_counts)
        if game.check_win() is not None or game.draw:
            break
        game.change_to_move()
    return game.check_win() or "draw"

def compare_mcts_iterations(configs=((50, 50), (100, 300), (300, 100)), games_per_config=5):
    rows = []
    for ix, io in configs:
        results = Counter(mcts_match(ix, io, seed=s) for s in range(games_per_config))
        rows.append({"iter_X": ix, "iter_O": io, "X_wins": results["X"], "O_wins": results["O"], "draws": results["draw"]})
    return rows

# -----------------------------
# Main jogavel
# -----------------------------

def main():
    print("=== PopOut ===")
    print("1 - Humano vs Humano")
    print("2 - Humano (X) vs IA (O)")
    print("3 - IA vs IA")
    modo = ""
    while modo not in ("1", "2", "3"):
        modo = input("Escolhe o modo [1/2/3]: ").strip()

    ai_x = MCTS(iterations=int(input("Iteracoes de X: "))) if modo == "3" else None
    ai_o = MCTS(iterations=int(input("Iteracoes de O: "))) if modo in ("2", "3") else None

    jogo = Poupout(6, 7)
    state_counts = defaultdict(int)
    state_counts[jogo.board_key()] += 1

    while True:
        jogo.display()
        winner = jogo.check_win()
        if winner is not None:
            print(f"Parabens, {winner} ganhou!")
            break
        if jogo.draw:
            print("Empate!")
            break

        ai_actual = ai_x if jogo.to_move == "X" else ai_o
        jogada = False
        while not jogada:
            if ai_actual is not None:
                print(f"IA ({jogo.to_move}) a pensar...")
                move = ai_actual.choose_move(jogo)
                if move is None:
                    print("Sem jogadas legais.")
                    return
                action, col = move
                print(f"  -> IA jogou: {action} coluna {col}")
                jogada = jogo.make_move(action, col)
            else:
                if jogo.repeated or jogo.check_full():
                    print("E possivel empatar o jogo [draw 0]")
                entrada = input(f"Jogador {jogo.to_move} [ex: put 3 / pop 2]: ").strip()
                try:
                    action, col_s = entrada.split()
                    jogada = jogo.make_move(action, int(col_s))
                    if not jogada:
                        print("Movimento invalido, tenta de novo.")
                except Exception:
                    print("Formato invalido. Usa: put 3 ou pop 2")

        jogo.check_repeat(state_counts)
        if jogo.check_win() is None and not jogo.draw:
            jogo.change_to_move()

if __name__ == "__main__":
    # Para jogar, descomenta a linha seguinte:
    main()
    pass

=== PopOut ===
1 - Humano vs Humano
2 - Humano (X) vs IA (O)
3 - IA vs IA


In [26]:
import random
import math
import copy
import csv
import time
import datetime
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

In [6]:
class Poupout:
    def __init__(self, rows, cols, moved="O", to_move="X"):
        #informações básicas do tabuleiro
        self.rows = rows                    
        self.cols = cols
        self.moved = moved
        self.to_move = to_move
        self.board = [["-" for _ in range(rows)] for _ in range(cols)]
        #informações adicionais para facilitar implementação de alguns métodos
        self.n_pieces = 0
        self.repeated = False
        self.last_move = None
        self.draw = False

    def display(self):
        print("  " + " ".join(str(i) for i in range(self.cols)))
        for r in range(self.rows):
            print("  ", end="")
            for c in range(self.cols):
                print(self.board[c][r], end="")
            print("")

    #usado para a criação do dataset
    def board_key(self):
        return tuple(tuple(col) for col in self.board), self.to_move

    def put(self, column):
        if column < 0 or column >= self.cols or self.board[column][0] != "-":
            raise Exception("Acao impossivel")
        for i in range(self.rows - 1, -1, -1):
            if self.board[column][i] == "-":
                self.board[column][i] = self.to_move
                self.n_pieces += 1
                return

    def pop(self, column):
        if column < 0 or column >= self.cols or self.board[column][-1] != self.to_move:
            raise Exception("Acao impossivel")
        self.board[column] = ["-"] + self.board[column][:-1]
        self.n_pieces -= 1

    def change_to_move(self):
        self.to_move, self.moved = self.moved, self.to_move

    def make_move(self, move, column):
        try:
            if move == "put":
                self.put(column)
            elif move == "pop":
                self.pop(column)
            elif move == "draw" and (self.repeated or self.check_full()):
                self.draw = True
            else:
                return False
            self.last_move = (move, column)
            return True
        except Exception:
            return False

    def check_row(self, row):
        line = "".join(self.board[c][row] for c in range(self.cols))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_col(self, col):
        line = "".join(self.board[col][r] for r in range(self.rows))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_diag1(self, row, col):
        cells = min(self.rows - row, self.cols - col)
        if cells < 4:
            return None
        line = "".join(self.board[col + i][row + i] for i in range(cells))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_diag2(self, row, col):
        cells = min(self.rows - row, col + 1)
        if cells < 4:
            return None
        line = "".join(self.board[col - i][row + i] for i in range(cells))
        if self.moved * 4 in line:
            return self.moved
        if self.to_move * 4 in line:
            return self.to_move
        return None

    def check_win(self):
        if self.last_move is None:
            return None
        move, column = self.last_move
        adversary_win = False

        if move == "put":
            #verificar o local da ultima peça colocada
            row = None
            for r in range(self.rows):
                if self.board[column][r] == self.moved:
                    row = r
                    break

            for f in [lambda: self.check_row(row), lambda: self.check_col(column),
                      lambda: self.check_diag1(max(0, row - column), max(0, column - row)),
                      lambda: self.check_diag2(max(0, row - (self.cols - 1 - column)), min(self.cols - 1, row + column))]:
                winner = f()
                if winner is not None:
                    return winner

        elif move == "pop":
            #verificar a coluna da peça retirada
            for row in range(self.rows - 1, -1, -1):
                if self.board[column][row] == "-":
                    break
                for f in [lambda row=row: self.check_row(row),
                          lambda row=row: self.check_diag1(max(0, row - column), max(0, column - row)),
                          lambda row=row: self.check_diag2(max(0, row - (self.cols - 1 - column)), min(self.cols - 1, row + column))]:
                    winner = f()
                    if winner == self.moved:
                        return winner
                    if winner == self.to_move:
                        adversary_win = True
        #apenas verificar se adversãrio ganhou depois do jogador que jogou
        if adversary_win:
            return self.to_move
        return None

    def check_full(self):
        return self.n_pieces == self.rows * self.cols

    def check_repeat(self, state_counts):
        key = self.board_key()
        state_counts[key] += 1
        if state_counts[key] >= 3:
            self.repeated = True
        return self.repeated

    #métodos auxiliares para a implementação de MCTS
    def clone(self):
        new = Poupout(self.rows, self.cols, self.moved, self.to_move)
        new.board = copy.deepcopy(self.board)
        new.n_pieces = self.n_pieces
        new.repeated = self.repeated
        new.last_move = self.last_move
        new.draw = self.draw
        return new

    def legal_moves(self):
        moves = []
        for c in range(self.cols):
            if self.board[c][0] == "-":
                moves.append(("put", c))
            if self.board[c][-1] == self.to_move:
                moves.append(("pop", c))
        if self.repeated or self.check_full():
            moves.append(("draw", 0))
        return moves

    def is_terminal(self):
        return self.check_win() is not None or self.draw

    def get_result(self, maximizing_player):
        winner = self.check_win()
        if winner == maximizing_player:
            return 1
        if winner is not None:
            return -1
        return 0


In [7]:
class MCTSNode:
    def __init__(self, game_state, parent=None, move=None):
        self.state = game_state
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0.0
        self.visits = 0
        self._untried = game_state.legal_moves()
        random.shuffle(self._untried)

    def is_fully_expanded(self):
        return len(self._untried) == 0

    def uct_score(self, c=math.sqrt(2)):
        if self.visits == 0:
            return float("inf")
        return self.wins / self.visits + c * math.sqrt(math.log(self.parent.visits) / self.visits)

    def best_child(self, c=math.sqrt(2)):
        return max(self.children, key=lambda n: n.uct_score(c))

class MCTS:
    def __init__(self, iterations=600, c=math.sqrt(2), max_depth=40, rollout_policy="random"):
        self.iterations = iterations
        self.c = c
        self.max_depth = max_depth
        self.rollout_policy = rollout_policy

    def choose_move(self, game_state):
        root = MCTSNode(game_state.clone())
        if not root._untried:
            return None
        original_player = game_state.to_move
        for _ in range(self.iterations):
            node = self._select(root)
            if not node.state.is_terminal():
                node = self._expand(node)
            result = self._simulate(node, original_player)
            self._backpropagate(node, result, original_player)
        return max(root.children, key=lambda n: n.visits).move

    def _select(self, node):
        while not node.state.is_terminal() and node.is_fully_expanded() and node.children:
            node = node.best_child(self.c)
        return node

    def _expand(self, node):
        if node._untried:
            move = node._untried.pop()
            new_game = node.state.clone()
            new_game.make_move(move[0], move[1])
            new_game.change_to_move()
            child = MCTSNode(new_game, parent=node, move=move)
            node.children.append(child)
            return child
        return node

    def _rollout_move(self, sim):
        moves = sim.legal_moves()
        if self.rollout_policy == "prefer_center":
            # estrategia simples: em empates prefere colunas centrais
            center = (sim.cols - 1) / 2
            moves.sort(key=lambda m: abs(m[1] - center))
            top = moves[:max(1, len(moves)//2)]
            return random.choice(top)
        return random.choice(moves)

    def _simulate(self, node, original_player):
        sim = node.state.clone()
        depth = 0
        while not sim.is_terminal() and depth < self.max_depth:
            moves = sim.legal_moves()
            if not moves:
                break
            action, col = self._rollout_move(sim)
            sim.make_move(action, col)
            sim.change_to_move()
            depth += 1
        return sim.get_result(original_player)

    def _backpropagate(self, node, result, original_player):
        while node is not None:
            node.visits += 1
            node_player = node.state.moved
            if node_player == original_player:
                node.wins += result
            else:
                node.wins -= result
            node = node.parent


In [ ]:
def play(player_X, player_O):
    ai_x = player_X
    ai_o = player_O

    jogo = Poupout(6, 7)
    state_counts = defaultdict(int)
    state_counts[jogo.board_key()] += 1

    while True:
        jogo.display()
        winner = jogo.check_win()
        if winner is not None:
            print(f"Parabens, {winner} ganhou!")
            break
        if jogo.draw:
            print("Empate!")
            break

        jogador = ai_x if jogo.to_move == "X" else ai_o
        print(f"IA ({jogo.to_move}) a pensar...")
        jogada = jogador.choose_move(jogo)
        action, col = jogada
        jogo.make_move(action, col)
        print(f"  -> IA jogou: {action} coluna {col}")
        state_counts[jogo.board_key()] += 1
        jogo.check_repeat(state_counts)
        jogo.change_to_move()

In [9]:
Iter_Jogador_X=10000
Iter_Jogador_O=10000
Jogador_X=MCTS(iterations=Iter_Jogador_X)
Jogador_O=MCTS(iterations=Iter_Jogador_O)
play(Jogador_X, Jogador_O)

  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  -------
IA (X) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  --X----
IA (O) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  --XO---
IA (X) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---X---
  --XO---
IA (O) a pensar...
  -> IA jogou: put coluna 4
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---X---
  --XOO--
IA (X) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  --XX---
  --XOO--
IA (O) a pensar...
  -> IA jogou: put coluna 5
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  --XX---
  --XOOO-
IA (X) a pensar...
  -> IA jogou: put coluna 6
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  --XX---
  --XOOOX
IA (O) a pensar...
  -> IA jogou: put coluna 4
  0 1 2 3 4 5 6


In [64]:
def state_to_features(game):
    row = {}
    for c in range(game.cols):
        for r in range(game.rows):
            row[f"c{c}_r{r}"] = game.board[c][r]
    row["to_move"] = game.to_move
    return row

def generate_popout_dataset(n_games=20, mcts_iterations=120, max_depth=40, rollout_policy="random", seed=0, outfile="popout_dataset.csv", show_progress=False):
    start=time.time()
    random.seed(seed)
    ai = MCTS(iterations=mcts_iterations, max_depth=max_depth, rollout_policy=rollout_policy)
    rows = []
    perc=0

    for g in range(n_games):
        if show_progress and (g/n_games)*100>=perc+10:
            perc+=10
            cur=time.time()
            elapsed=cur-start
            expected=(elapsed/perc)*100
            left=expected-elapsed
            print(f"{perc}%  Faltam {str(datetime.timedelta(seconds=left))}")
        game = Poupout(6, 7)
        state_counts = defaultdict(int)
        state_counts[game.board_key()] += 1

        while True:
            if game.is_terminal():
                break
            move = ai.choose_move(game)
            features = state_to_features(game)
            features["move"] = f"{move[0]}_{move[1]}"
            rows.append(features)
            game.make_move(move[0], move[1])
            game.check_repeat(state_counts)
            game.change_to_move()

        fieldnames = list(rows[0].keys())
        with open(outfile, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
    if show_progress:
        print("Done")
    

def load_csv_dataset(path, label_col="label"):
    with open(path) as file:
        ds=csv.DictReader(file)
        rows=[]
        for r in ds:
            rows.append(r)
        return rows

In [36]:
print("Dataset de 100 iterações:")
generate_popout_dataset(n_games=500, mcts_iterations=100, outfile="popout_dataset_100.csv", show_progress=True)
print()
print("Dataset de 1000 iterações:")
generate_popout_dataset(n_games=200, mcts_iterations=1000, outfile="popout_dataset_1000.csv", show_progress=True)
print()
print("Dataset de 10000 iterações:")
generate_popout_dataset(n_games=100, mcts_iterations=10000, outfile="popout_dataset_10000.csv", show_progress=True)

Dataset de 100 iterações:
10%  Faltam 0:03:35.813885
20%  Faltam 0:03:24.229897
30%  Faltam 0:02:57.824198
40%  Faltam 0:02:33.479711
50%  Faltam 0:02:08.596974
60%  Faltam 0:01:41.749196
70%  Faltam 0:01:12.052775
80%  Faltam 0:00:48.802509
90%  Faltam 0:00:24.813716
Done

Dataset de 1000 iterações:
10%  Faltam 0:15:23.159969
20%  Faltam 0:13:40.500233
30%  Faltam 0:12:45.745478
40%  Faltam 0:10:36.317876
50%  Faltam 0:08:42.270149
60%  Faltam 0:06:48.426291
70%  Faltam 0:05:12.153876
80%  Faltam 0:03:31.146701
90%  Faltam 0:01:47.309064
Done

Dataset de 10000 iterações:
10%  Faltam 1:33:14.922618
20%  Faltam 1:24:45.788389
30%  Faltam 1:12:46.621040
40%  Faltam 1:00:32.772124
50%  Faltam 0:50:57.790189
60%  Faltam 0:40:40.507501
70%  Faltam 0:30:26.869652
80%  Faltam 0:19:38.303901
90%  Faltam 0:09:55.999920
Done


In [ ]:
@dataclass
class ID3Node:
    prediction: Any
    attribute: Optional[str] = None
    children: Optional[Dict[Any, "ID3Node"]] = None
    is_leaf: bool = False

    def classify(self, example):
        if self.is_leaf or self.attribute is None:
            return self.prediction
        value = example.get(self.attribute)
        if self.children and value in self.children:
            return self.children[value].classify(example)
        return self.prediction

    def pretty(self, indent=""):
        if self.is_leaf:
            return indent + f"Classe: {self.prediction}\n"
        s = indent + f"Se {self.attribute}:\n"
        for value, child in sorted(self.children.items(), key=lambda x: str(x[0])):
            s += indent + f"  = {value} ->\n"
            s += child.pretty(indent + "    ")
        return s

class ID3DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.label_col = None

    @staticmethod
    def entropy(labels):
        total = len(labels)
        counts = Counter(labels)
        return -sum((cnt/total) * math.log2(cnt/total) for cnt in counts.values() if cnt)

    def information_gain(self, rows, attribute, label_col):
        base = self.entropy([r[label_col] for r in rows])
        total = len(rows)
        parts = defaultdict(list)
        for r in rows:
            parts[r[attribute]].append(r)
        remainder = sum((len(part)/total) * self.entropy([r[label_col] for r in part]) for part in parts.values())
        return base - remainder

    def majority_label(self, rows, label_col):
        return Counter(r[label_col] for r in rows).most_common(1)[0][0]

    def fit(self, rows, label_col="label", attributes=None):
        if not rows:
            raise ValueError("Dataset vazio")
        self.label_col = label_col
        if attributes is None:
            attributes = [a for a in rows[0].keys() if a != label_col]
        self.root = self._id3(rows, attributes, label_col, depth=0)
        return self

    def _id3(self, rows, attributes, label_col, depth):
        labels = [r[label_col] for r in rows]
        majority = self.majority_label(rows, label_col)
        if len(set(labels)) == 1:
            return ID3Node(prediction=labels[0], is_leaf=True)
        if not attributes or len(rows) < self.min_samples_split or (self.max_depth is not None and depth >= self.max_depth):
            return ID3Node(prediction=majority, is_leaf=True)

        gains = [(self.information_gain(rows, a, label_col), a) for a in attributes]
        best_gain, best_attr = max(gains, key=lambda x: x[0])
        if best_gain <= 1e-12:
            return ID3Node(prediction=majority, is_leaf=True)

        node = ID3Node(prediction=majority, attribute=best_attr, children={}, is_leaf=False)
        values = sorted(set(r[best_attr] for r in rows), key=str)
        remaining = [a for a in attributes if a != best_attr]
        for v in values:
            subset = [r for r in rows if r[best_attr] == v]
            node.children[v] = self._id3(subset, remaining, label_col, depth + 1)
        return node

    def predict_one(self, example):
        if self.root is None:
            raise ValueError("A arvore ainda nao foi treinada")
        return self.root.classify(example)

    def predict(self, rows):
        return [self.predict_one(r) for r in rows]

    def score(self, rows, label_col=None):
        label_col = label_col or self.label_col
        preds = self.predict(rows)
        return sum(p == r[label_col] for p, r in zip(preds, rows)) / len(rows)

    def pretty(self):
        return self.root.pretty() if self.root else "<arvore vazia>"

In [65]:
def train_test_split(rows, test_ratio=0.25, seed=0):
    random.Random(seed).shuffle(rows)
    cut = int(len(rows) * (1 - test_ratio))
    return rows[:cut], rows[cut:]

def discretize_numeric_dataset(rows, label_col="label", bins=3):
    # Discretizacao por quantis: reduz tamanho da arvore e funciona bem no Iris.
    attrs = [a for a in rows[0] if a != label_col]
    thresholds = {}
    for a in attrs:
        vals = sorted(float(r[a]) for r in rows)
        qs = []
        for b in range(1, bins):
            idx = int(len(vals) * b / bins)
            qs.append(vals[idx])
        thresholds[a] = qs

    new_rows = []
    for r in rows:
        nr = {label_col: r[label_col]}
        for a in attrs:
            x = float(r[a])
            k = 0
            while k < len(thresholds[a]) and x > thresholds[a][k]:
                k += 1
            nr[a] = f"bin{k}"
        new_rows.append(nr)
    return new_rows, thresholds

In [99]:
class ID3DecisionTreePlayer:
    def __init__(self, tree):
        self.tree=tree

    def choose_move(self, state):
        game=state_to_features(state)
        move=self.tree.predict_one(game)
        move=move.split("_")
        return (move[0], int(move[1]))

In [100]:
dataset_100=load_csv_dataset(path="popout_dataset_100.csv", label_col="move")
tree = ID3DecisionTree(max_depth=5)
tree.fit(dataset_100, label_col="move")
Player=ID3DecisionTreePlayer(tree)
jogo=Poupout(6,7)
jogo.display()
print(Player.choose_move(jogo))

  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  -------
('put', 3)


In [101]:
dataset_10000=load_csv_dataset(path="popout_dataset_10000.csv", label_col="move")
tree = ID3DecisionTree(max_depth=5)
tree.fit(dataset_100, label_col="move")
Player_X=ID3DecisionTreePlayer(tree)
Player_O=MCTS(iterations=1000)
play(Player_X, Player_O)

  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  -------
IA (X) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  -------
  ---X---
IA (O) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  ---X---
IA (X) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  --XX---
IA (O) a pensar...
  -> IA jogou: put coluna 1
  0 1 2 3 4 5 6
  -------
  -------
  -------
  -------
  ---O---
  -OXX---
IA (X) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  -------
  ---X---
  ---O---
  -OXX---
IA (O) a pensar...
  -> IA jogou: put coluna 2
  0 1 2 3 4 5 6
  -------
  -------
  -------
  ---X---
  --OO---
  -OXX---
IA (X) a pensar...
  -> IA jogou: put coluna 3
  0 1 2 3 4 5 6
  -------
  -------
  ---X---
  ---X---
  --OO---
  -OXX---
IA (O) a pensar...
  -> IA jogou: put coluna 1
  0 1 2 3 4 5 6


TypeError: list indices must be integers or slices, not NoneType

## Exemplo: gerar dataset PopOut

A célula seguinte gera um dataset pequeno. Para a entrega final, aumentar `n_games` e `mcts_iterations`, por exemplo `n_games=100` e `mcts_iterations=300`.

In [ ]:
# Exemplo rapido
rows = generate_popout_dataset(n_games=5, mcts_iterations=50, seed=1, outfile="popout_dataset.csv")
print("Exemplos gerados:", len(rows))
print(rows[0] if rows else "Dataset vazio")

## Treinar e avaliar ID3 no dataset PopOut

In [ ]:
if rows:
    results = evaluate_popout_tree(rows, max_depth_values=(3, 5, 7, None), seed=1)
    for r in results:
        print(r)

    train, test = train_test_split(rows, test_ratio=0.25, seed=1)
    tree = ID3DecisionTree(max_depth=5).fit(train, label_col="label")
    print("Excerto da arvore:")
    print(tree.pretty()[:3000])

## Iris: carregar, discretizar e treinar

Coloca o ficheiro `iris.csv` na mesma pasta do notebook com as colunas: `sepal_length`, `sepal_width`, `petal_length`, `petal_width`, `label`.

In [103]:
# Exemplo para quando tiveres o ficheiro iris.csv
iris_rows, _ = load_csv_dataset("iris.csv", label_col="label")
iris_disc, thresholds = discretize_numeric_dataset(iris_rows, label_col="label", bins=3)
train, test = train_test_split(iris_disc, test_ratio=0.25, seed=2)
iris_tree = ID3DecisionTree(max_depth=None).fit(train, label_col="label")
print("Accuracy treino:", iris_tree.score(train, "label"))
print("Accuracy teste:", iris_tree.score(test, "label"))
print("Thresholds usados:", thresholds)
print(iris_tree.pretty())

FileNotFoundError: [Errno 2] No such file or directory: 'iris.csv'

## Experiência MCTS: número de iterações

Compara jogadores MCTS com diferentes orçamentos de iterações.

In [ ]:
# Exemplo pequeno; aumentar games_per_config para resultados mais estáveis
comparison = compare_mcts_iterations(configs=((50, 50), (50, 150), (150, 50)), games_per_config=3)
for row in comparison:
    print(row)

## Conclusões esperadas

- Mais iterações MCTS tendem a produzir jogadas mais estáveis, mas aumentam o tempo de decisão.
- A árvore ID3 aprende uma aproximação da política produzida pelo MCTS.
- Para PopOut, limitar a profundidade da árvore pode melhorar a generalização e reduzir overfitting.
- No Iris, a discretização por quantis reduz o tamanho da árvore e evita ramos demasiado específicos.